# 实验报告03：正则表达式与网络结构化数据

## 一、实验环境与提交要求

### 1.1 身份与环境记录

In [1]:
import hashlib
import os
import platform
import uuid
import sys

STUDENT_ID = 20244080107

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")
print(f"[Python] {sys.version.split()[0]}")

[Conda 环境] python课程
[学号] 20244080107
[机器指纹] 025bce21d5564943
[RUN_ID] 20244080107-025bce21d5564943
[Python] 3.14.7


这段代码记录了当前 conda 环境、学号、机器指纹、运行标识和 Python 版本。机器指纹由当前电脑的信息生成，因此在不同电脑上可能不同。实验要求使用独立的 conda 环境，不能使用 `base`。

## 二、实验内容与过程

### 实验1：正则表达式的基础匹配

#### 1.1 匹配数字、空白和完整单词

In [2]:
import re

text = "cat on mat; later: cats and dogs"

print("数字：", re.findall(r"\d+", text))
print("空白：", re.findall(r"\s+", text))
print("完整单词 cat：", re.findall(r"\bcat\b", text))

数字： []
空白： [' ', ' ', ' ', ' ', ' ', ' ']
完整单词 cat： ['cat']


`\d+` 匹配一个或多个数字；原文本中没有数字，因此结果为空列表。`\s+` 匹配一个或多个连续空白字符，本例中找到 6 个空格。`\bcat\b` 使用单词边界限定完整单词，所以匹配到独立的 `cat`，但不会匹配 `cats` 中的 `cat`。

#### 1.2 使用选择符并比较匹配顺序

In [3]:
text_animals = "cat dog bird CAT"

print("匹配 cat 或 dog：", re.findall(r"cat|dog", text_animals))

text_a = "aaaa"
print("短选项在前：", re.findall(r"a|aa|aaa", text_a))
print("长选项在前：", re.findall(r"aaa|aa|a", text_a))

匹配 cat 或 dog： ['cat', 'dog']
短选项在前： ['a', 'a', 'a', 'a']
长选项在前： ['aaa', 'a']


`|` 表示“或”，因此 `cat|dog` 会匹配其中任意一个选项。`re.findall()` 从左到右扫描文本，并按正则表达式中选项的书写顺序尝试匹配：`a|aa|aaa` 总是先匹配单个 `a`；`aaa|aa|a` 则优先匹配较长的 `aaa`。因此，调整选择项顺序会改变匹配结果。

#### 1.3 使用重复次数量词

In [4]:
text_quantifiers = "ct cat caat caaat"

patterns = [
    r"ca*t",
    r"ca+t",
    r"ca?t",
    r"ca{2}t",
    r"ca{1,2}t",
]

for pattern in patterns:
    print(f"{pattern}: {re.findall(pattern, text_quantifiers)}")

ca*t: ['ct', 'cat', 'caat', 'caaat']
ca+t: ['cat', 'caat', 'caaat']
ca?t: ['ct', 'cat']
ca{2}t: ['caat']
ca{1,2}t: ['cat', 'caat']


`*` 表示前一个字符出现 0 次或多次；`+` 表示出现 1 次或多次；`?` 表示出现 0 次或 1 次；`{2}` 表示恰好出现 2 次；`{1,2}` 表示出现 1 到 2 次。因此，`ca*t` 可以匹配没有 `a` 的 `ct`，而 `ca+t` 不可以；`ca{2}t` 只匹配 `caat`。

### 实验2：分组、回溯引用与替换

In [5]:
import re

# 1. 提取邮箱的用户名和域名
email_text = "联系：alice@example.com；备选：bob@test.org"
email_pattern = r"([\w.+-]+)@([\w.-]+\.[A-Za-z]{2,})"
emails = re.findall(email_pattern, email_text)

print("邮箱用户名和域名：", emails)

# 2. 查找连续重复的单词
repeat_text = "cat cat, dog cat, bird bird"
repeat_pattern = r"\b(\w+)\s+\1\b"
repeated_words = re.findall(repeat_pattern, repeat_text)

print("连续重复的单词：", repeated_words)

# 3. 将日期转换为“月-日-年”
date_text = "2024-05-01\n2024/05/02\n2024.05.03"
date_pattern = r"\b(\d{4})[-/.](\d{2})[-/.](\d{2})\b"
converted_dates = re.sub(date_pattern, r"\2-\3-\1", date_text)

print("\n转换前的日期：")
print(date_text)
print("\n转换后的日期：")
print(converted_dates)
print("转换前后行数：", len(date_text.splitlines()), len(converted_dates.splitlines()))

# 4. 统一电话号码格式
phone_text = "联系我：123-456-7890 或 987.654.3210"
phone_pattern = r"(?<!\d)(\d{3})[-.](\d{3})[-.](\d{4})(?!\d)"
formatted_phones = re.sub(phone_pattern, r"(\1) \2-\3", phone_text)

print("\n统一格式后的电话号码：")
print(formatted_phones)

邮箱用户名和域名： [('alice', 'example.com'), ('bob', 'test.org')]
连续重复的单词： ['cat', 'bird']

转换前的日期：
2024-05-01
2024/05/02
2024.05.03

转换后的日期：
05-01-2024
05-02-2024
05-03-2024
转换前后行数： 3 3

统一格式后的电话号码：
联系我：(123) 456-7890 或 (987) 654-3210


邮箱正则使用捕获组分别提取用户名和域名。重复单词正则中的 `\1` 是后向引用，要求后一个单词与第一个捕获组完全相同。日期正则捕获年、月、日，再通过替换表达式按“月-日-年”顺序输出；转换前后行数应相同。电话号码正则识别三段数字及其连字符或句点，并统一替换成 `(区号) 电话号码` 的格式。

### 实验3：使用 BeautifulSoup 解析 HTML

In [7]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")

# 页面标题与 h1
title = soup.title
heading = soup.find("h1")

print("页面标题：", title.get_text(strip=True))
print("h1 标签名：", heading.name)
print("h1 文本：", heading.get_text(strip=True))

# 所有课程段落
course_paragraphs = soup.find_all("p", class_="course")
print("\n课程段落：")
for paragraph in course_paragraphs:
    print(paragraph.get_text(strip=True))

# 所有链接及其 href
links = soup.find_all("a")
print("\n链接：")
for link in links:
    print(f"文字：{link.get_text(strip=True)}，href：{link.get('href')}")

# 比较整个文档、单个段落和单个链接的文本
print("\n整个页面的可见文本：")
print(soup.get_text(separator=" ", strip=True))
print("单个段落的文本：", course_paragraphs[0].get_text(strip=True))
print("单个链接的文本：", links[0].get_text(strip=True))

页面标题： 实验课程
h1 标签名： h1
h1 文本： 数据分析

课程段落：
第六讲：正则表达式
第七讲：结构化数据

链接：
文字：讲义六，href：https://example.org/lecture06
文字：讲义七，href：https://example.org/lecture07

整个页面的可见文本：
实验课程 数据分析 第六讲：正则表达式 第七讲：结构化数据 讲义六 讲义七
单个段落的文本： 第六讲：正则表达式
单个链接的文本： 讲义六


BeautifulSoup 使用 `html.parser` 将 HTML 字符串解析为树形结构。`title` 和 `find("h1")` 用于读取标题及其标签信息；`find_all()` 找到所有课程段落和链接，链接的地址通过 `href` 属性取得。`get_text()` 可以提取页面整体或指定标签中的可见文本。